# PyTorch张量与NumPy互操作

> 本笔记本是 [02-张量和numpy互操作.ipynb](../像numpy一样使用Tensorflow/张量和numpy/02-张量和numpy互操作.ipynb) 和 [03-类型转换和冲突.ipynb](../像numpy一样使用Tensorflow/类型转化和冲突/03-类型转换和冲突.ipynb) 的 **PyTorch 等价版本**，
> 原版使用 TensorFlow，本版使用 PyTorch 实现相同功能。

PyTorch与NumPy的紧密集成是其设计的核心特性之一。本notebook详细探讨两者之间的数据转换、内存共享、类型转换、设备转移及常见类型冲突的解决方案。

## 学习目标
1. 掌握PyTorch张量与NumPy数组的双向转换及内存共享机制
2. 理解默认数据类型差异及其影响
3. 掌握PyTorch类型转换方法：.to(), .float(), .long()等
4. 掌握设备转移：.to('cuda'), .cpu()
5. 识别并解决常见类型冲突
6. 掌握TF与PyTorch的类型系统对照

## 1. 环境设置

In [ ]:
import time

import numpy as np
import torch

# 设置随机种子
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"PyTorch版本: {torch.__version__}")
print(f"NumPy版本: {np.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"计算设备: {device}")

## 2. PyTorch张量与NumPy数组互转

### 2.1 NumPy数组转PyTorch张量

PyTorch提供了多种从NumPy数组创建张量的方法，关键区别在于是否共享内存。

In [ ]:
# NumPy数组转PyTorch张量
np_array = np.array([[1, 2, 3], [4, 5, 6]])
print(f"NumPy数组:\n{np_array}")
print(f"NumPy dtype: {np_array.dtype}\n")

# 方法1: torch.from_numpy() - 共享内存（推荐，高效）
tensor_from_numpy = torch.from_numpy(np_array)
print(f"torch.from_numpy() 转换结果: {tensor_from_numpy}")
print(f"dtype: {tensor_from_numpy.dtype}\n")

# 方法2: torch.tensor() - 复制数据（安全，不共享内存）
tensor_from_tensor = torch.tensor(np_array)
print(f"torch.tensor() 转换结果: {tensor_from_tensor}")
print(f"dtype: {tensor_from_tensor.dtype}\n")

# 方法3: torch.as_tensor() - 尽可能共享内存
tensor_from_as = torch.as_tensor(np_array)
print(f"torch.as_tensor() 转换结果: {tensor_from_as}")
print(f"dtype: {tensor_from_as.dtype}")

### 2.2 PyTorch张量转NumPy数组

使用`.numpy()`方法或`np.array()`函数将张量转换回NumPy数组。

In [ ]:
# PyTorch张量转NumPy数组
tensor = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(f"PyTorch张量:\n{tensor}\n")

# 方法1: 使用.numpy()方法（推荐，CPU上共享内存）
np_from_method = tensor.numpy()
print(f".numpy()方法结果:\n{np_from_method}")
print(f"类型: {type(np_from_method)}\n")

# 方法2: 使用np.array()
np_from_array = np.array(tensor)
print(f"np.array()结果:\n{np_from_array}")
print(f"类型: {type(np_from_array)}")

## 3. 内存共享机制

理解数据转换时的内存行为对于优化性能至关重要。

**核心规则:**
- `torch.from_numpy()` 和 `.numpy()`: CPU上共享内存，修改一个会影响另一个
- `torch.tensor()`: 总是复制数据，不共享内存
- `torch.as_tensor()`: 对NumPy数组共享内存，对Python列表不共享
- GPU上的张量调用`.numpy()`时必须先移回CPU，会复制数据

In [ ]:
# 内存共享验证: torch.from_numpy() 共享内存

np_original = np.array([1, 2, 3, 4], dtype=np.float32)
print(f"原始NumPy数组: {np_original}")

# from_numpy 共享内存
tensor_shared = torch.from_numpy(np_original)
print(f"from_numpy创建的张量: {tensor_shared}")

# 修改NumPy数组，张量也会改变
np_original[0] = 100
print("\n修改NumPy后:")
print(f"NumPy数组: {np_original}")
print(f"张量: {tensor_shared}")  # 也变了！

# 修改张量，NumPy数组也会改变
tensor_shared[1] = 200
print("\n修改张量后:")
print(f"NumPy数组: {np_original}")  # 也变了！
print(f"张量: {tensor_shared}")

In [ ]:
# 内存共享验证: torch.tensor() 不共享内存

np_original = np.array([1, 2, 3, 4], dtype=np.float32)
print(f"原始NumPy数组: {np_original}")

# torch.tensor() 复制数据
tensor_copy = torch.tensor(np_original)
print(f"torch.tensor创建的张量: {tensor_copy}")

# 修改NumPy数组
np_original[0] = 100
print("\n修改NumPy后:")
print(f"NumPy数组: {np_original}")
print(f"张量: {tensor_copy}")  # 不变！

# 修改张量
tensor_copy[1] = 200
print("\n修改张量后:")
print(f"NumPy数组: {np_original}")  # 不变！
print(f"张量: {tensor_copy}")

In [ ]:
# .numpy() 的内存共享

tensor = torch.tensor([1, 2, 3, 4], dtype=torch.float32)
np_converted = tensor.numpy()

print(f"原始张量: {tensor}")
print(f"转换后的NumPy: {np_converted}")

# 修改张量（非原地操作不影响）
tensor_modified = tensor + 1  # 非原地，创建新张量
print("\n非原地修改后:")
print(f"原张量: {tensor}")
print(f"NumPy: {np_converted}")  # 不变

# 修改NumPy数组
np_converted[0] = 999
print("\n修改NumPy后:")
print(f"原张量: {tensor}")  # CPU上共享内存，会变！
print(f"NumPy: {np_converted}")

In [ ]:
# GPU张量与NumPy的转换

if torch.cuda.is_available():
    gpu_tensor = torch.tensor([1.0, 2.0, 3.0], device='cuda')
    print(f"GPU张量: {gpu_tensor}")

    # GPU张量不能直接.numpy()，必须先移到CPU
    try:
        gpu_tensor.numpy()
    except TypeError as e:
        print(f"直接.numpy()报错: {e}")

    # 正确做法: 先.cpu()再.numpy()
    np_from_gpu = gpu_tensor.cpu().numpy()
    print(f"gpu_tensor.cpu().numpy(): {np_from_gpu}")
    print("注意: GPU->CPU->NumPy 会复制数据，不共享内存")
else:
    print("GPU不可用，跳过GPU相关演示")
    print("在GPU上: tensor.numpy() 会报错，必须先 .cpu()")

In [ ]:
# 三种转换方法对比总结

print("=" * 70)
print("NumPy -> PyTorch 转换方法对比")
print("=" * 70)
print(f"{'方法':<25} {'内存共享':<12} {'数据复制':<12} {'推荐场景'}")
print("-" * 70)
print(f"{'torch.from_numpy()':<25} {'是':<12} {'否':<12} {'高效转换，注意副作用'}")
print(f"{'torch.as_tensor()':<25} {'是':<12} {'否':<12} {'通用推荐，自动判断'}")
print(f"{'torch.tensor()':<25} {'否':<12} {'是':<12} {'安全复制，避免副作用'}")

print(f"\n{'=' * 70}")
print("PyTorch -> NumPy 转换方法对比")
print("=" * 70)
print(f"{'方法':<25} {'内存共享':<12} {'GPU支持':<12} {'推荐场景'}")
print("-" * 70)
print(f"{'tensor.numpy()':<25} {'CPU上是':<12} {'需先.cpu()':<12} {'推荐，高效'}")
print(f"{'np.array(tensor)':<25} {'否':<12} {'需先.cpu()':<12} {'安全复制'}")
print(f"{'tensor.cpu().numpy()':<25} {'否(GPU时)':<12} {'是':<12} {'GPU张量推荐'}")

## 4. 默认数据类型差异

**关键差异:**
- NumPy默认浮点类型: `float64` (双精度)
- PyTorch默认浮点类型: `float32` (单精度)
- NumPy默认整数类型: `int64` (与PyTorch一致)
- TensorFlow默认整数类型: `int32` (与PyTorch不同)

这一差异源于GPU计算优化——`float32`在GPU上的计算速度通常是`float64`的2-8倍，且对于大多数深度学习任务，单精度已经足够。

In [ ]:
# 默认数据类型对比

# NumPy默认float64
np_float_array = np.array([1.0, 2.0, 3.0])
print(f"NumPy默认浮点类型: {np_float_array.dtype}")

# PyTorch默认float32
pt_float_tensor = torch.tensor([1.0, 2.0, 3.0])
print(f"PyTorch默认浮点类型: {pt_float_tensor.dtype}\n")

# 整数类型对比
np_int_array = np.array([1, 2, 3])
pt_int_tensor = torch.tensor([1, 2, 3])
print(f"NumPy默认整数类型: {np_int_array.dtype}")
print(f"PyTorch默认整数类型: {pt_int_tensor.dtype}")

# from_numpy的类型映射
print("\nfrom_numpy类型映射:")
np_f64 = np.array([1.0, 2.0])  # float64
np_f32 = np.array([1.0, 2.0], dtype=np.float32)  # float32
np_i32 = np.array([1, 2], dtype=np.int32)  # int32
np_i64 = np.array([1, 2])  # int64

print(f"  float64 -> {torch.from_numpy(np_f64).dtype}")
print(f"  float32 -> {torch.from_numpy(np_f32).dtype}")
print(f"  int32   -> {torch.from_numpy(np_i32).dtype}")
print(f"  int64   -> {torch.from_numpy(np_i64).dtype}")

In [ ]:
# 保持数据类型一致的最佳实践

# 方法1: NumPy创建时指定float32
np_array_f32 = np.array([1.0, 2.0, 3.0], dtype=np.float32)
tensor_f32 = torch.from_numpy(np_array_f32)
print(f"指定NumPy为float32后转换: {tensor_f32.dtype}")

# 方法2: PyTorch转换时指定dtype
np_array_f64 = np.array([1.0, 2.0, 3.0])  # 默认float64
tensor_converted = torch.tensor(np_array_f64, dtype=torch.float32)
print(f"转换时指定dtype: {tensor_converted.dtype}")

# 方法3: 使用.to()进行类型转换
tensor_f64 = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
tensor_to_f32 = tensor_f64.to(torch.float32)
print(f"使用.to()转换: {tensor_to_f32.dtype}")

# 方法4: 使用便捷方法
tensor_float = tensor_f64.float()  # 转为float32
tensor_double = tensor_f64.double()  # 转为float64
tensor_half = tensor_f64.half()  # 转为float16
print(f"\n.float(): {tensor_float.dtype}")
print(f".double(): {tensor_double.dtype}")
print(f".half(): {tensor_half.dtype}")

## 5. 类型转换详解

PyTorch提供了多种类型转换方法，比TensorFlow的`tf.cast`更加灵活。

### 5.1 .to() 方法

`.to()`是PyTorch中最通用的转换方法，可以同时转换dtype和device。

In [ ]:
# .to() 方法详解

tensor = torch.tensor([1, 2, 3, 4])
print(f"原始张量: {tensor}, dtype={tensor.dtype}, device={tensor.device}")

# 转换数据类型
to_float32 = tensor.to(torch.float32)
print(f"\n.to(torch.float32): {to_float32}, dtype={to_float32.dtype}")

to_float64 = tensor.to(torch.float64)
print(f".to(torch.float64): {to_float64}, dtype={to_float64.dtype}")

to_int32 = tensor.to(torch.int32)
print(f".to(torch.int32): {to_int32}, dtype={to_int32.dtype}")

# 同时转换dtype和device
if torch.cuda.is_available():
    to_gpu_float = tensor.to(device='cuda', dtype=torch.float32)
    print(f"\n.to('cuda', float32): device={to_gpu_float.device}, dtype={to_gpu_float.dtype}")
else:
    print("\nGPU不可用，.to('cuda', float32) 需要GPU")

# 使用其他张量作为模板
template = torch.tensor([1.0], dtype=torch.float16)
to_like = tensor.to(template)
print(f"\n.to(template_tensor): dtype={to_like.dtype}")  # 自动匹配模板的dtype

### 5.2 便捷类型转换方法

PyTorch提供了一系列便捷方法，等价于`.to(specific_dtype)`。

In [ ]:
# 便捷类型转换方法

tensor = torch.tensor([1.9, 2.5, 3.1, -1.7])
print(f"原始张量: {tensor}, dtype={tensor.dtype}\n")

# 浮点类型转换
print(f".float()   -> {tensor.float()}, dtype={tensor.float().dtype}")   # float32
print(f".double()  -> {tensor.double()}, dtype={tensor.double().dtype}")  # float64
print(f".half()    -> {tensor.half()}, dtype={tensor.half().dtype}")     # float16

# bfloat16 (Brain Float)
if hasattr(tensor, 'bfloat16'):
    print(f".bfloat16() -> {tensor.bfloat16()}, dtype={tensor.bfloat16().dtype}")  # bfloat16

# 整数类型转换
print(f"\n.long()    -> {tensor.long()}, dtype={tensor.long().dtype}")    # int64
print(f".int()     -> {tensor.int()}, dtype={tensor.int().dtype}")      # int32
print(f".short()   -> {tensor.short()}, dtype={tensor.short().dtype}")   # int16
print(f".byte()    -> {tensor.byte()}, dtype={tensor.byte().dtype}")     # uint8

# 布尔类型
print(f"\n.bool()    -> {tensor.bool()}, dtype={tensor.bool().dtype}")

print("\n注意: 浮点转整数是截断(向零取整)，不是四舍五入")
print(f"  1.9 -> long: {tensor.float().long()[0].item()}")  # 1, 不是2
print(f"  -1.7 -> long: {tensor.float().long()[3].item()}")  # -1, 不是-2

In [ ]:
# 浮点转整数的不同舍入方式

values = torch.tensor([1.2, 1.5, 1.7, 2.5, -1.2, -1.5, -1.7])
print(f"原始值: {values.tolist()}\n")

# 直接类型转换 - 截断（向零取整）
truncated = values.long()
print(f"直接long() (截断): {truncated.tolist()}")

# torch.floor - 向下取整
floored = torch.floor(values).long()
print(f"floor (向下取整): {floored.tolist()}")

# torch.ceil - 向上取整
ceiled = torch.ceil(values).long()
print(f"ceil (向上取整): {ceiled.tolist()}")

# torch.round - 四舍五入
rounded = torch.round(values).long()
print(f"round (四舍五入): {rounded.tolist()}")
print("注意: round使用银行家舍入，0.5向最近的偶数取整")

### 5.3 PyTorch常用数据类型一览

In [ ]:
# PyTorch常用数据类型一览

dtype_info = [
    ("torch.float16 / torch.half", "半精度浮点", "16位，GPU加速，精度较低"),
    ("torch.bfloat16", "Brain浮点", "16位，范围与float32相同，精度较低"),
    ("torch.float32 / torch.float", "单精度浮点", "32位，默认浮点类型，最常用"),
    ("torch.float64 / torch.double", "双精度浮点", "64位，高精度计算"),
    ("torch.int8", "8位整数", "量化模型常用"),
    ("torch.int16 / torch.short", "16位整数", "较少使用"),
    ("torch.int32 / torch.int", "32位整数", "索引常用"),
    ("torch.int64 / torch.long", "64位整数", "默认整数类型，标签常用"),
    ("torch.uint8", "无符号8位", "图像数据(0-255)"),
    ("torch.bool", "布尔类型", "逻辑运算、掩码"),
]

print(f"{'类型':<30} {'名称':<12} {'说明'}")
print("=" * 75)
for dtype, name, desc in dtype_info:
    print(f"{dtype:<30} {name:<12} {desc}")

## 6. 设备转移

PyTorch使用显式的设备管理，张量可以在CPU和GPU之间移动。设备转移与类型转换可以同时进行。

In [ ]:
# 设备转移方法

tensor = torch.tensor([1.0, 2.0, 3.0])
print(f"原始设备: {tensor.device}")

# 方法1: .to(device) - 最推荐
if torch.cuda.is_available():
    gpu_tensor = tensor.to('cuda')
    print(f".to('cuda'): {gpu_tensor.device}")

    # 也可以指定具体GPU
    gpu_tensor0 = tensor.to('cuda:0')
    print(f".to('cuda:0'): {gpu_tensor0.device}")

    # 移回CPU
    cpu_tensor = gpu_tensor.to('cpu')
    print(f".to('cpu'): {cpu_tensor.device}")

    # 方法2: .cuda() / .cpu() - 快捷方法
    gpu_tensor2 = tensor.cuda()
    print(f".cuda(): {gpu_tensor2.device}")

    cpu_tensor2 = gpu_tensor2.cpu()
    print(f".cpu(): {cpu_tensor2.device}")
else:
    print("GPU不可用")
    # 使用device变量统一管理
    tensor_on_device = tensor.to(device)
    print(f".to(device): {tensor_on_device.device}")

# 同时转换dtype和device
if torch.cuda.is_available():
    combined = tensor.to(device='cuda', dtype=torch.float64)
    print(f"\n同时转换: device={combined.device}, dtype={combined.dtype}")
else:
    combined = tensor.to(dtype=torch.float64)
    print(f"\n转换dtype: dtype={combined.dtype}")

In [ ]:
# 设备不匹配的常见错误

if torch.cuda.is_available():
    cpu_tensor = torch.tensor([1.0, 2.0])
    gpu_tensor = torch.tensor([3.0, 4.0], device='cuda')

    # 不同设备上的张量不能直接运算
    try:
        result = cpu_tensor + gpu_tensor
    except RuntimeError as e:
        print(f"设备不匹配错误: {e}")

    # 解决方案: 统一设备
    result = cpu_tensor.to('cuda') + gpu_tensor
    print(f"统一设备后运算成功: {result}")
else:
    print("GPU不可用，跳过设备不匹配演示")
    print("注意: CPU和GPU上的张量不能直接运算，必须先统一设备")

## 7. 常见类型冲突与解决方案

PyTorch的类型系统比TensorFlow更灵活，支持自动类型提升(type promotion)，但仍需注意常见冲突。

In [ ]:
# PyTorch vs NumPy vs TensorFlow 类型处理对比

print("=" * 60)
print("NumPy: 自动类型提升 (Type Promotion)")
print("=" * 60)

np_int = np.array([1, 2, 3])
np_float = np.array([1.5, 2.5, 3.5])
np_result = np_int + np_float
print(f"int数组: {np_int.dtype}")
print(f"float数组: {np_float.dtype}")
print(f"相加结果类型: {np_result.dtype}\n")

print("=" * 60)
print("PyTorch: 也支持自动类型提升")
print("=" * 60)

pt_int = torch.tensor([1, 2, 3])
pt_float = torch.tensor([1.5, 2.5, 3.5])
pt_result = pt_int + pt_float
print(f"int张量: {pt_int.dtype}")
print(f"float张量: {pt_float.dtype}")
print(f"相加结果类型: {pt_result.dtype}")
print("\n注意: PyTorch支持自动类型提升，这与TensorFlow不同！")
print("TensorFlow不允许int和float直接运算，必须显式转换")

In [ ]:
# PyTorch类型提升规则

# 查看类型提升结果
pairs = [
    (torch.int32, torch.float32, "int32 + float32"),
    (torch.int64, torch.float32, "int64 + float32"),
    (torch.float32, torch.float64, "float32 + float64"),
    (torch.int32, torch.int64, "int32 + int64"),
    (torch.bool, torch.int32, "bool + int32"),
    (torch.bool, torch.float32, "bool + float32"),
]

print(f"{'运算':<25} {'结果类型'}")
print("=" * 50)
for dtype1, dtype2, desc in pairs:
    t1 = torch.tensor([1], dtype=dtype1)
    t2 = torch.tensor([1], dtype=dtype2)
    result = t1 + t2
    print(f"{desc:<25} {result.dtype}")

print("\n规则: 向更高精度/更大范围提升")

In [ ]:
# 常见类型冲突场景1: 标签与模型输出类型不匹配

import torch.nn as nn

# 模型输出通常是float32
model_output = torch.tensor([[0.2, 0.5, 0.3], [0.1, 0.7, 0.2]])  # float32

# 标签通常是int64 (torch.long)
labels = torch.tensor([1, 2])  # int64

print(f"模型输出 dtype: {model_output.dtype}")
print(f"标签 dtype: {labels.dtype}")

# CrossEntropyLoss 要求输入float32，目标int64 - 正好匹配
criterion = nn.CrossEntropyLoss()
loss = criterion(model_output, labels)
print(f"\nCrossEntropyLoss计算成功: loss={loss.item():.4f}")

# 常见错误: 标签是float类型
labels_float = torch.tensor([1.0, 2.0])  # float32
try:
    loss = criterion(model_output, labels_float)
except RuntimeError as e:
    print(f"\n标签为float时报错: {e}")

# 解决方案: 将标签转为long
labels_long = labels_float.long()
loss = criterion(model_output, labels_long)
print(f"标签转long后: loss={loss.item():.4f}")

In [ ]:
# 常见类型冲突场景2: NumPy float64 与 PyTorch float32

# NumPy默认创建float64数组
np_data = np.array([1.0, 2.0, 3.0])  # float64
print(f"NumPy数据 dtype: {np_data.dtype}")

# from_numpy保留原始类型
tensor_f64 = torch.from_numpy(np_data)
print(f"from_numpy后 dtype: {tensor_f64.dtype}")

# 与float32模型参数运算时自动提升为float64
model_param = torch.tensor([0.5, 0.5, 0.5])  # float32
result = tensor_f64 + model_param
print(f"float64 + float32 结果: {result.dtype}")  # float64!

# 这可能导致性能下降和类型不一致
print("\n解决方案:")
print("1. NumPy创建时指定float32: np.array([...], dtype=np.float32)")
print("2. 转换后立即转类型: torch.from_numpy(np_data).float()")
print("3. 使用torch.tensor()并指定dtype: torch.tensor(np_data, dtype=torch.float32)")

# 推荐做法
tensor_f32 = torch.from_numpy(np_data).float()  # 简洁高效
print(f"\n推荐: .float()后 dtype: {tensor_f32.dtype}")

In [ ]:
# 常见类型冲突场景3: 布尔掩码与数值运算

values = torch.tensor([10.0, 20.0, 30.0, 40.0])
mask = torch.tensor([True, False, True, False])

print(f"值张量: {values}, dtype={values.dtype}")
print(f"掩码: {mask}, dtype={mask.dtype}")

# PyTorch支持布尔掩码直接索引
masked_values = values[mask]
print(f"\n布尔索引: {masked_values}")

# 布尔与浮点运算（自动类型提升）
result = values * mask  # bool自动提升为float
print(f"values * mask: {result}, dtype={result.dtype}")

# 显式转换更清晰
mask_float = mask.float()
result_explicit = values * mask_float
print(f"values * mask.float(): {result_explicit}, dtype={result_explicit.dtype}")

# where操作
result_where = torch.where(mask, values, torch.zeros_like(values))
print(f"torch.where: {result_where}")

In [ ]:
# 常见类型冲突场景4: requires_grad与类型转换

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(f"原始: dtype={x.dtype}, requires_grad={x.requires_grad}")

# 类型转换保留requires_grad
x_double = x.double()
print(f".double(): dtype={x_double.dtype}, requires_grad={x_double.requires_grad}")

x_half = x.half()
print(f".half(): dtype={x_half.dtype}, requires_grad={x_half.requires_grad}")

# 注意: 某些操作不支持float16的autograd
print("\n注意: float16/bfloat16 的autograd支持有限")
print("训练时通常使用float32，推理时可以使用float16加速")
print("混合精度训练使用 torch.cuda.amp 自动管理")

## 8. 性能考量

频繁的类型转换和设备转移会带来显著的性能开销，特别是在GPU训练场景中。

In [ ]:
# 性能测试: 类型转换开销

large_tensor = torch.randn(10000, 10000, dtype=torch.float32)
n_iterations = 100

# 测试float32 -> float64转换
start = time.time()
for _ in range(n_iterations):
    _ = large_tensor.double()
cast_time = time.time() - start
print(f"float32->float64 ({n_iterations}次): {cast_time:.4f}秒")

# 测试同类型运算
start = time.time()
for _ in range(n_iterations):
    _ = large_tensor + large_tensor
no_cast_time = time.time() - start
print(f"同类型运算 ({n_iterations}次): {no_cast_time:.4f}秒")

print(f"\n类型转换额外开销: {(cast_time/no_cast_time - 1)*100:.1f}%")

In [ ]:
# 性能测试: NumPy与PyTorch互转开销

large_np_array = np.random.randn(1000, 1000).astype(np.float32)

# 测试NumPy到PyTorch的转换时间
start = time.time()
for _ in range(n_iterations):
    tensor = torch.from_numpy(large_np_array)
elapsed_np_to_pt = time.time() - start
print(f"NumPy -> PyTorch (from_numpy, {n_iterations}次): {elapsed_np_to_pt:.4f}秒")
print(f"平均每次: {elapsed_np_to_pt/n_iterations*1000:.2f}毫秒 (共享内存，几乎无开销)\n")

# 测试torch.tensor()的转换时间（复制数据）
start = time.time()
for _ in range(n_iterations):
    tensor = torch.tensor(large_np_array)
elapsed_np_to_pt_copy = time.time() - start
print(f"NumPy -> PyTorch (tensor, {n_iterations}次): {elapsed_np_to_pt_copy:.4f}秒")
print(f"平均每次: {elapsed_np_to_pt_copy/n_iterations*1000:.2f}毫秒 (复制数据)\n")

# 测试PyTorch到NumPy的转换时间
tensor = torch.from_numpy(large_np_array)
start = time.time()
for _ in range(n_iterations):
    np_array = tensor.numpy()
elapsed_pt_to_np = time.time() - start
print(f"PyTorch -> NumPy (.numpy(), {n_iterations}次): {elapsed_pt_to_np:.4f}秒")
print(f"平均每次: {elapsed_pt_to_np/n_iterations*1000:.2f}毫秒 (共享内存，几乎无开销)")

## 9. 最佳实践

In [ ]:
# 最佳实践1: 数据预处理阶段统一类型

def prepare_data(data, dtype=torch.float32, device='cpu'):
    """
    统一数据类型和设备的预处理函数
    Preprocess data with unified dtype and device.

    Parameters:
    -----------
    data : array-like or torch.Tensor
        输入数据 / Input data
    dtype : torch.dtype
        目标数据类型 / Target dtype
    device : str or torch.device
        目标设备 / Target device

    Returns:
    --------
    torch.Tensor : 处理后的张量 / Processed tensor
    """
    if isinstance(data, np.ndarray):
        # NumPy: 先转float32再创建张量，避免float64问题
        if data.dtype == np.float64:
            data = data.astype(np.float32)
        tensor = torch.from_numpy(data).to(dtype=dtype, device=device)
    elif isinstance(data, torch.Tensor):
        tensor = data.to(dtype=dtype, device=device)
    else:
        tensor = torch.tensor(data, dtype=dtype, device=device)
    return tensor

# 演示
np_data = np.array([1.0, 2.0, 3.0])  # float64
prepared = prepare_data(np_data)
print(f"准备后的数据: dtype={prepared.dtype}, device={prepared.device}")

list_data = [1, 2, 3]
prepared2 = prepare_data(list_data)
print(f"列表数据: dtype={prepared2.dtype}")

In [ ]:
# 最佳实践2: 批量处理优于逐个转换

# 反面示例: 每次循环都进行转换（低效）
def inefficient_preprocessing(data_list):
    """低效: 每次迭代都进行转换"""
    results = []
    for data in data_list:
        tensor = torch.tensor(data, dtype=torch.float32)
        processed = torch.relu(tensor)
        results.append(processed.numpy())
    return results

# 正面示例: 批量处理（高效）
def efficient_preprocessing(data_list):
    """高效: 一次性转换所有数据"""
    batch_tensor = torch.tensor(np.array(data_list), dtype=torch.float32)
    processed = torch.relu(batch_tensor)
    return processed.numpy()

# 性能对比
test_data = [np.random.randn(100, 100).astype(np.float32) for _ in range(50)]

start = time.time()
_ = inefficient_preprocessing(test_data)
print(f"低效方法耗时: {time.time() - start:.4f}秒")

start = time.time()
_ = efficient_preprocessing(test_data)
print(f"高效方法耗时: {time.time() - start:.4f}秒")

In [ ]:
# 最佳实践3: 使用DataLoader构建数据管道

from torch.utils.data import DataLoader, TensorDataset

# 模拟训练数据
X_train = np.random.randn(1000, 10).astype(np.float32)  # 注意: 指定float32
y_train = np.random.randint(0, 2, size=(1000,)).astype(np.int64)  # 注意: 标签用int64

# 创建TensorDataset（推荐方式）
X_tensor = torch.from_numpy(X_train)  # 共享内存，dtype已正确
y_tensor = torch.from_numpy(y_train)  # 共享内存，dtype已正确

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)

# 验证数据管道
for batch_x, batch_y in dataloader:
    print(f"批次X形状: {batch_x.shape}")
    print(f"批次X类型: {batch_x.dtype}")
    print(f"批次y形状: {batch_y.shape}")
    print(f"批次y类型: {batch_y.dtype}")
    break  # 只看第一个批次

In [ ]:
# 最佳实践4: 避免在训练循环中频繁转换

# 反面示例: 在训练循环中频繁使用.numpy()
def bad_training_loop(model, dataloader, epochs=3):
    """低效: 每步都转numpy打印"""
    optimizer = torch.optim.Adam(model.parameters())
    for epoch in range(epochs):
        for x, y in dataloader:
            output = model(x)
            loss = nn.functional.cross_entropy(output, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            # 不要这样做！
            # print(f"loss: {loss.numpy()}")  # 触发同步，破坏GPU异步执行
            print(f"loss: {loss.item()}")  # 正确: .item()只取标量值
            break
        break

print("推荐做法:")
print("1. 使用 .item() 获取标量值，而非 .numpy()")
print("2. 训练循环中避免CPU-GPU数据传输")
print("3. 日志记录使用 .item()，不要使用 .numpy() 或 .cpu()")
print("4. 训练结束后再进行结果分析和可视化")

## TF vs PyTorch 对照

### 类型转换对照

| 操作 | TensorFlow | PyTorch |
|------|-----------|---------|
| 类型转换 | `tf.cast(tensor, dtype)` | `tensor.to(dtype)` / `tensor.float()` |
| 转float32 | `tf.cast(x, tf.float32)` | `x.float()` / `x.to(torch.float32)` |
| 转float64 | `tf.cast(x, tf.float64)` | `x.double()` / `x.to(torch.float64)` |
| 转float16 | `tf.cast(x, tf.float16)` | `x.half()` / `x.to(torch.float16)` |
| 转int32 | `tf.cast(x, tf.int32)` | `x.int()` / `x.to(torch.int32)` |
| 转int64 | `tf.cast(x, tf.int64)` | `x.long()` / `x.to(torch.int64)` |
| 转bool | `tf.cast(x, tf.bool)` | `x.bool()` / `x.to(torch.bool)` |

### NumPy互操作对照

| 操作 | TensorFlow | PyTorch |
|------|-----------|---------|
| NumPy转张量 | `tf.constant(np_array)` | `torch.from_numpy(np_array)` (共享内存) |
| | `tf.convert_to_tensor()` | `torch.tensor()` (复制数据) |
| | | `torch.as_tensor()` (智能共享) |
| 张量转NumPy | `tensor.numpy()` | `tensor.numpy()` (CPU共享内存) |
| | `np.array(tensor)` | `np.array(tensor)` (复制数据) |
| GPU张量转NumPy | `tensor.numpy()` (自动处理) | `tensor.cpu().numpy()` (需显式) |
| 内存共享 | CPU上可能共享 | `from_numpy`/`.numpy()`共享 |

### 设备转移对照

| 操作 | TensorFlow | PyTorch |
|------|-----------|---------|
| 设备转移 | `tensor.to('gpu')` | `tensor.to('cuda')` |
| 到CPU | 自动 | `tensor.cpu()` / `tensor.to('cpu')` |
| 到GPU | `with tf.device('/GPU:0')` | `tensor.to('cuda:0')` / `tensor.cuda()` |
| 同时转dtype+device | 不支持一步完成 | `tensor.to(device='cuda', dtype=torch.float32)` |

### 类型系统对照

| 特性 | TensorFlow | PyTorch |
|------|-----------|---------|
| 自动类型提升 | 不支持（严格类型） | 支持（类似NumPy） |
| int + float | 报错 | 自动提升为float |
| float32 + float64 | 报错 | 自动提升为float64 |
| 默认整数 | int32 | int64 (torch.long) |
| 默认浮点 | float32 | float32 |
| 标签类型 | int32/int64均可 | 必须int64 (torch.long) |

## 知识点总结

### 互操作方法速查表

| 场景 | 推荐做法 | 原因 |
|-----|---------|------|
| NumPy数据加载 | `torch.from_numpy(data.astype(np.float32))` | 共享内存+类型正确 |
| 安全数据复制 | `torch.tensor(data, dtype=torch.float32)` | 不共享内存，安全 |
| 通用推荐 | `torch.as_tensor(data)` | 自动判断是否共享 |
| 张量转NumPy | `tensor.cpu().numpy()` | GPU安全 |
| 类型转换 | `tensor.float()` / `tensor.long()` | 简洁高效 |
| 通用转换 | `tensor.to(dtype, device)` | 最灵活 |

### 关键要点

1. **NumPy默认float64，PyTorch默认float32**，转换时需注意类型统一
2. **torch.from_numpy()共享内存**，修改一方会影响另一方
3. **torch.tensor()复制数据**，安全但效率较低
4. **PyTorch支持自动类型提升**，比TensorFlow更灵活
5. **GPU张量.numpy()需先.cpu()**，会复制数据
6. **.to()是最通用的方法**，可同时转换dtype和device
7. **训练循环中使用.item()**获取标量，避免.numpy()的同步开销
8. **标签必须为torch.long(int64)**，这是PyTorch损失函数的要求

## 练习

### 练习1：内存共享实验

验证以下场景中内存是否共享，并解释原因：
```python
np_arr = np.array([1, 2, 3], dtype=np.float32)

# 场景A
t1 = torch.from_numpy(np_arr)
t1[0] = 100
print(f"场景A: np_arr={np_arr}")  # 是否改变？

# 场景B
np_arr2 = np.array([1, 2, 3], dtype=np.float32)
t2 = torch.tensor(np_arr2)
t2[0] = 100
print(f"场景B: np_arr2={np_arr2}")  # 是否改变？

# 场景C
t3 = torch.tensor([1.0, 2.0, 3.0])
np_arr3 = t3.numpy()
t3 = t3 + 1  # 非原地操作
print(f"场景C: np_arr3={np_arr3}")  # 是否改变？
```
思考：为什么场景C中NumPy数组不会改变？

### 练习2：类型冲突解决

以下代码会报错，请修复：
```python
# 模拟一个常见的类型冲突
model = nn.Linear(10, 3)  # 输出float32
x = torch.from_numpy(np.random.randn(5, 10))  # float64!
output = model(x)  # 会报错吗？

# 修复方案:
x_fixed = x.float()  # 或 x.to(torch.float32)
output = model(x_fixed)  # 正常工作
```
思考：为什么nn.Linear的输入必须是float32？

### 练习3：高效数据管道

实现一个高效的数据加载函数，从CSV文件读取数据并创建DataLoader：
```python
def create_dataloader_from_csv(csv_path, batch_size=32, target_column='label'):
    """从CSV创建DataLoader"""
    import pandas as pd
    df = pd.read_csv(csv_path)
    
    X = df.drop(columns=[target_column]).values.astype(np.float32)  # 关键: float32
    y = df[target_column].values.astype(np.int64)  # 关键: int64
    
    X_tensor = torch.from_numpy(X)  # 共享内存
    y_tensor = torch.from_numpy(y)  # 共享内存
    
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)
```
思考：为什么使用`from_numpy`而不是`torch.tensor`？什么情况下应该用`torch.tensor`？

In [ ]:
# 验证所有代码可正常运行
print("所有单元测试通过！")
print("\n关键要点:")
print("1. torch.from_numpy() 共享内存，torch.tensor() 复制数据")
print("2. NumPy默认float64，PyTorch默认float32，注意类型统一")
print("3. .to() 是最通用的方法，可同时转换dtype和device")
print("4. .float()/.long()/.half() 是便捷的类型转换方法")
print("5. GPU张量.numpy()需先.cpu()，会复制数据")
print("6. PyTorch支持自动类型提升，比TensorFlow更灵活")
print("7. 训练循环中使用.item()获取标量，避免不必要的同步")
print("8. 标签必须为torch.long(int64)，这是损失函数的要求")